# Gullak: BKT knowledge tracing — SYNTHETIC ONLY

Increment 1: validate quiz answer rows and generate 96 synthetic answers (12 learners, four submissions, two questions).
Increment 2: simulate synthetic learners from known BKT parameters, fit per-concept BKT by EM on a learner-separated training split, check held-out next-answer prediction against a constant baseline, and export a draft parameter file labelled synthetic. No real-student accuracy. No app integration; quiz graph pass marks remain the only unlock rule.

Run cells top to bottom on a Colab CPU runtime (Runtime → Run all); no installs, keys, Drive mount or network needed. It takes well under a minute. The last cell downloads `knowledge-tracing-params.synthetic-draft.json`.

The nine input columns match the latest R3 contract: event_id, learner_id, topic_id, concept_id, question_id, correct, answered_at, attempt_no, content_version. No question_index column. Content version is SHA-256 of exact topic-content UTF-8 bytes. Use an immutable content archive supplied by R3 to obtain question order and concepts; never reconstruct history from the current answer key.

Each submission shares attempt_no (the topic's quiz_attempts after submission). Within it, questions are ordered by position in pinned content. Missing historical content or ambiguous positions fail closed. The synthetic content below is an engineering fixture with the real money-in / money-out concept IDs, not a reviewed lesson.

Synthetic labels live in notebook and dataset metadata, not extra input columns. Keep future real learner rows out of this publicly shared notebook. Story safe/risky/harmful verdicts are excluded from BKT for now.

In [ ]:
import hashlib
import json
import random
import re
from datetime import datetime, timedelta, timezone
from uuid import UUID

LABEL = "SYNTHETIC — NOT REAL STUDENT DATA"
FIELDS = {"event_id", "learner_id", "topic_id", "concept_id", "question_id",
          "correct", "answered_at", "attempt_no", "content_version"}
# Synthetic teaching fixture, NOT an approved/published money-basics lesson.
SYNTHETIC_CONTENT = {
    "topicId": "money-basics", "locale": "en",
    "title": "SYNTHETIC engineering fixture", "summary": LABEL,
    "variants": [{"id": "synthetic-check", "format": "quiz", "questions": [
        {"id": "money-in", "conceptId": "money-in", "prompt": "Gift arrives: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "in", "explanation": "A gift adds money."},
        {"id": "money-out", "conceptId": "money-out", "prompt": "Paying a bill: money in or out?",
         "options": [{"id": "in", "text": "In"}, {"id": "out", "text": "Out"}],
         "correctOptionId": "out", "explanation": "A payment takes money away."}
    ]}]
}
# Hash these exact UTF-8 bytes. Never reserialize real content to calculate its hash.
SYNTHETIC_CONTENT_RAW = json.dumps(SYNTHETIC_CONTENT, ensure_ascii=False, separators=(",", ":"))


def synthetic_content_hash():
    return hashlib.sha256(SYNTHETIC_CONTENT_RAW.encode("utf-8")).hexdigest()


def validate_answer_rows(rows, registry):
    """Validate nine-field rows against a trusted, version-pinned question registry.
    Registry keys: (topic_id, content_version, question_id).
    Values: {"concept_id": canonical slug, "position": content question order}.
    Return sorted copies; no guessing, grading, mastery update or mutation.
    """
    if not isinstance(rows, list) or not rows:
        raise ValueError("expected a non-empty list of quiz answers")
    parsed, event_ids, positions, questions, versions = [], set(), set(), set(), {}
    for index, raw in enumerate(rows):
        if not isinstance(raw, dict) or set(raw) != FIELDS:
            raise ValueError(f"row {index}: expected exactly the nine quiz_answers fields")
        row = dict(raw)
        for field in ("event_id", "learner_id"):
            value = row[field]
            if not isinstance(value, str):
                raise ValueError(f"row {index}: invalid {field}")
            try:
                normalized = str(UUID(value))
            except ValueError as error:
                raise ValueError(f"row {index}: invalid {field}") from error
            row[field] = normalized
        for field in ("topic_id", "concept_id", "question_id"):
            if not isinstance(row[field], str) or not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", row[field]):
                raise ValueError(f"row {index}: invalid {field} slug")
        if type(row["correct"]) is not bool:
            raise ValueError(f"row {index}: correct must be boolean")
        if type(row["attempt_no"]) is not int or not 1 <= row["attempt_no"] <= 2147483647:
            raise ValueError(f"row {index}: attempt_no must be a positive Postgres integer")
        if not isinstance(row["content_version"], str) or not re.fullmatch(r"[a-f0-9]{64}", row["content_version"]):
            raise ValueError(f"row {index}: content_version must be lowercase SHA-256 hex")
        if not isinstance(row["answered_at"], str):
            raise ValueError(f"row {index}: timestamp must be text")
        try:
            stamp = datetime.fromisoformat(row["answered_at"].replace("Z", "+00:00"))
        except ValueError as error:
            raise ValueError(f"row {index}: invalid timestamp") from error
        if stamp.tzinfo is None or stamp.utcoffset() is None:
            raise ValueError(f"row {index}: timestamp must include a timezone")
        row["answered_at"] = stamp.astimezone(timezone.utc).isoformat()
        definition = registry.get((row["topic_id"], row["content_version"], row["question_id"]))
        if not isinstance(definition, dict) or definition.get("concept_id") != row["concept_id"]:
            raise ValueError(f"row {index}: unknown pinned question or concept mismatch")
        position = definition.get("position")
        if type(position) is not int or position < 0:
            raise ValueError(f"row {index}: missing trusted question position")
        submission = (row["learner_id"], row["topic_id"], row["attempt_no"])
        if submission in versions and versions[submission] != row["content_version"]:
            raise ValueError(f"row {index}: mixed content versions within a submission")
        versions[submission] = row["content_version"]
        position_key = (*submission, position)
        question_key = (*submission, row["question_id"])
        if row["event_id"] in event_ids or position_key in positions or question_key in questions:
            raise ValueError(f"row {index}: duplicate event, question or ambiguous content order")
        event_ids.add(row["event_id"])
        positions.add(position_key)
        questions.add(question_key)
        parsed.append((row, position))
    parsed.sort(key=lambda pair: (pair[0]["learner_id"], pair[0]["topic_id"], pair[0]["attempt_no"], pair[1]))
    # A whole submission may share a server timestamp. Check between submissions,
    # not within one: content order, rather than timestamp, orders its answers.
    ranges = {}
    for row, _ in parsed:
        key = (row["learner_id"], row["topic_id"], row["attempt_no"])
        stamp = datetime.fromisoformat(row["answered_at"])
        old = ranges.get(key, (stamp, stamp))
        ranges[key] = (min(old[0], stamp), max(old[1], stamp))
    previous = {}
    for (learner, topic, _), (earliest, latest) in sorted(ranges.items()):
        key = (learner, topic)
        if key in previous and earliest < previous[key]:
            raise ValueError("timestamps contradict quiz attempt order")
        previous[key] = latest
    return [row for row, _ in parsed]


def generate_synthetic_responses(seed=42):
    """Fixed small engineering fixture; no model training or accuracy claims."""
    if type(seed) is not int:
        raise ValueError("seed must be an integer")
    rng = random.Random(seed)
    version = synthetic_content_hash()
    quiz = SYNTHETIC_CONTENT["variants"][0]["questions"]
    registry = {("money-basics", version, q["id"]):
                {"concept_id": q["conceptId"], "position": position}
                for position, q in enumerate(quiz)}
    rows = []
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    for learner_index in range(12):
        learner = str(UUID(int=rng.getrandbits(128), version=4))
        for attempt in range(1, 5):
            timestamp = (start + timedelta(days=attempt - 1, minutes=learner_index)).isoformat()
            for question in quiz:
                rows.append({
                    "event_id": str(UUID(int=rng.getrandbits(128), version=4)),
                    "learner_id": learner, "topic_id": "money-basics",
                    "concept_id": question["conceptId"], "question_id": question["id"],
                    "correct": rng.random() < 0.25 + attempt * 0.12,
                    "answered_at": timestamp, "attempt_no": attempt, "content_version": version,
                })
    metadata = {"label": LABEL, "synthetic": True, "production_approved": False,
                "seed": seed, "content_version": version,
                "generator": "fixed engineering fixture; not a calibrated learner model"}
    return rows, registry, metadata


In [ ]:
import copy
import unittest

class SyntheticInputTests(unittest.TestCase):
    def setUp(self):
        self.rows, self.registry, self.metadata = generate_synthetic_responses(seed=42)

    def test_valid_rows_match_exact_contract(self):
        result = validate_answer_rows(self.rows, self.registry)
        self.assertEqual(len(result), 96)
        self.assertEqual(set(result[0]), {
            "event_id", "learner_id", "topic_id", "concept_id", "question_id",
            "correct", "answered_at", "attempt_no", "content_version",
        })

    def test_seed_is_reproducible_and_metadata_is_synthetic(self):
        self.assertEqual(generate_synthetic_responses(seed=42), (self.rows, self.registry, self.metadata))
        self.assertNotEqual(generate_synthetic_responses(seed=43)[0], self.rows)
        self.assertTrue(self.metadata["synthetic"])
        self.assertFalse(self.metadata["production_approved"])
        self.assertIn("SYNTHETIC", self.metadata["label"])
        self.assertEqual(self.metadata["seed"], 42)

    def test_each_required_field_is_checked(self):
        for field in self.rows[0]:
            with self.subTest(field=field):
                rows = copy.deepcopy(self.rows)
                del rows[0][field]
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_invalid_values_are_rejected(self):
        cases = [
            ("event_id", "not-a-uuid"), ("learner_id", "not-a-uuid"),
            ("topic_id", "Money Basics"), ("concept_id", "Money In"),
            ("question_id", "Question_1"), ("correct", 1), ("correct", "false"),
            ("answered_at", "2026-01-01T00:00:00"), ("answered_at", "not-a-date"),
            ("attempt_no", 0), ("attempt_no", True), ("attempt_no", 1.5),
            ("content_version", "synthetic-v1"), ("content_version", "g" * 64),
        ]
        for field, value in cases:
            with self.subTest(field=field, value=value):
                rows = copy.deepcopy(self.rows)
                rows[0][field] = value
                with self.assertRaises(ValueError):
                    validate_answer_rows(rows, self.registry)

    def test_unknown_fields_and_empty_input_are_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["name"] = "private"
        for invalid in [rows, [], None, [None]]:
            with self.subTest(invalid=type(invalid).__name__):
                with self.assertRaises(ValueError):
                    validate_answer_rows(invalid, self.registry)

    def test_obsolete_question_index_column_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["question_index"] = 0
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_invalid_or_ambiguous_registry_positions_are_rejected(self):
        key = ("money-basics", self.rows[1]["content_version"], "money-out")
        for position in [None, -1, True, 0]:
            registry = copy.deepcopy(self.registry)
            registry[key]["position"] = position
            with self.subTest(position=position), self.assertRaises(ValueError):
                validate_answer_rows(self.rows, registry)

    def test_postgres_integer_overflow_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[0]["attempt_no"] = 2147483648
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_boolean_seed_is_not_accepted_as_integer(self):
        with self.assertRaises(ValueError):
            generate_synthetic_responses(seed=True)

    def test_duplicate_event_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["event_id"] = rows[0]["event_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_duplicate_question_within_submission_is_rejected(self):
        rows = copy.deepcopy(self.rows)
        rows[1]["question_id"] = rows[0]["question_id"]
        rows[1]["concept_id"] = rows[0]["concept_id"]
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_unknown_version_question_or_concept_is_rejected(self):
        for field, value in [("content_version", "0" * 64), ("question_id", "unknown"),
                             ("concept_id", "money-out"), ("topic_id", "unknown")]:
            rows = copy.deepcopy(self.rows)
            rows[0][field] = value
            with self.subTest(field=field), self.assertRaises(ValueError):
                validate_answer_rows(rows, self.registry)

    def test_version_is_a_hash_of_the_synthetic_content(self):
        self.assertEqual(self.rows[0]["content_version"], self.metadata["content_version"])
        self.assertEqual(self.metadata["content_version"], synthetic_content_hash())

    def test_a_submission_cannot_mix_versions(self):
        rows = copy.deepcopy(self.rows)
        other_version = "1" * 64
        registry = dict(self.registry)
        registry[("money-basics", other_version, "money-out")] = {"concept_id": "money-out", "position": 1}
        rows[1]["content_version"] = other_version
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, registry)

    def test_pinned_content_orders_equal_timestamps_without_uuid_tiebreak(self):
        rows = list(reversed(self.rows))
        before = copy.deepcopy(rows)
        result = validate_answer_rows(rows, self.registry)
        self.assertEqual(rows, before)
        for left, right in zip(result, result[1:]):
            key = lambda r: (r["learner_id"], r["topic_id"], r["attempt_no"],
                             self.registry[(r["topic_id"], r["content_version"], r["question_id"])]["position"])
            self.assertLess(key(left), key(right))

    def test_timestamp_is_normalized_to_utc(self):
        rows = copy.deepcopy(self.rows)
        for row in rows:
            row["answered_at"] = row["answered_at"].replace("+00:00", "Z")
        self.assertTrue(all(row["answered_at"].endswith("+00:00")
                            for row in validate_answer_rows(rows, self.registry)))

    def test_time_cannot_go_backwards_in_attempt_order(self):
        rows = copy.deepcopy(self.rows)
        rows[2]["answered_at"] = "2025-01-01T00:00:00+00:00"
        with self.assertRaises(ValueError):
            validate_answer_rows(rows, self.registry)

    def test_repeated_questions_across_attempts_are_valid(self):
        self.assertEqual(self.rows[0]["question_id"], self.rows[2]["question_id"])
        self.assertNotEqual(self.rows[0]["attempt_no"], self.rows[2]["attempt_no"])
        validate_answer_rows(self.rows, self.registry)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(SyntheticInputTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise AssertionError('SYNTHETIC input validation tests failed')
print('SYNTHETIC input checks passed. No training performed.')


In [ ]:
rows, registry, dataset_metadata = generate_synthetic_responses(seed=42)
validated_rows = validate_answer_rows(rows, registry)
print(LABEL)
print(json.dumps(dataset_metadata, indent=2, ensure_ascii=False))
print(f"SYNTHETIC: {len(validated_rows)} answer events, 12 fictional learners")
print(json.dumps(validated_rows[:2], indent=2))


## Increment 2: fit BKT on synthetic learners

Each concept gets four parameters: P(init) knows it before the first question, P(learn) learns it between attempts, P(slip) answers wrong despite knowing it, P(guess) answers right without knowing it. No forgetting; guess and slip are bounded so knowing stays distinguishable from not knowing.

Synthetic learners are drawn from known parameters, so the fit can be checked for **recovery** (true vs fitted) and for held-out **next-answer prediction** against always predicting the training correct rate. Learners are split by hash, never by row, so no learner is in both sets. These numbers say the fitter works, not how well real students are modelled.

In [ ]:
import math

# Standard BKT bounds: no forgetting, and guess + slip < 1 so "knows it" stays
# distinguishable from "doesn't". Guess allows up to 0.5 for two-option questions.
BKT_BOUNDS = {"p_init": (0.01, 0.99), "p_learn": (0.01, 0.5),
              "p_slip": (0.01, 0.3), "p_guess": (0.01, 0.5)}
# Hidden parameters the synthetic learners are simulated from, so the fit can be
# checked for recovery. Invented for engineering; not estimates of real students.
SYNTHETIC_TRUE_PARAMS = {
    "money-in": {"p_init": 0.25, "p_learn": 0.20, "p_slip": 0.08, "p_guess": 0.25},
    "money-out": {"p_init": 0.15, "p_learn": 0.12, "p_slip": 0.10, "p_guess": 0.20},
}
MASTERY_THRESHOLD = 0.95


def _check_params(params):
    if not isinstance(params, dict) or set(params) != set(BKT_BOUNDS):
        raise ValueError("expected exactly p_init, p_learn, p_slip and p_guess")
    for name, (low, high) in BKT_BOUNDS.items():
        value = params[name]
        if type(value) not in (int, float) or not low <= value <= high:
            raise ValueError(f"{name} must be between {low} and {high}")


def simulate_bkt_learners(true_params=None, n_learners=400, n_attempts=6, seed=7):
    """Rows in the nine-field contract, drawn from a known BKT process. SYNTHETIC."""
    true_params = SYNTHETIC_TRUE_PARAMS if true_params is None else true_params
    if type(seed) is not int or type(n_learners) is not int or type(n_attempts) is not int:
        raise ValueError("seed, n_learners and n_attempts must be integers")
    if n_learners < 1 or n_attempts < 1:
        raise ValueError("need at least one learner and one attempt")
    quiz = SYNTHETIC_CONTENT["variants"][0]["questions"]
    if set(true_params) != {q["conceptId"] for q in quiz}:
        raise ValueError("true_params must cover exactly the fixture's concepts")
    for params in true_params.values():
        _check_params(params)
    rng = random.Random(seed)
    version = synthetic_content_hash()
    registry = {("money-basics", version, q["id"]): {"concept_id": q["conceptId"], "position": position}
                for position, q in enumerate(quiz)}
    rows = []
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    for learner_index in range(n_learners):
        learner = str(UUID(int=rng.getrandbits(128), version=4))
        knows = {c: rng.random() < p["p_init"] for c, p in true_params.items()}
        for attempt in range(1, n_attempts + 1):
            stamp = (start + timedelta(days=attempt - 1, seconds=learner_index)).isoformat()
            for question in quiz:
                p = true_params[question["conceptId"]]
                known = knows[question["conceptId"]]
                correct = rng.random() < ((1 - p["p_slip"]) if known else p["p_guess"])
                rows.append({
                    "event_id": str(UUID(int=rng.getrandbits(128), version=4)),
                    "learner_id": learner, "topic_id": "money-basics",
                    "concept_id": question["conceptId"], "question_id": question["id"],
                    "correct": correct, "answered_at": stamp,
                    "attempt_no": attempt, "content_version": version,
                })
                if not known and rng.random() < p["p_learn"]:
                    knows[question["conceptId"]] = True
    metadata = {"label": LABEL, "synthetic": True, "production_approved": False,
                "seed": seed, "learners": n_learners, "attempts": n_attempts,
                "content_version": version, "true_params": true_params,
                "generator": "known BKT process; tests parameter recovery, not real learning"}
    return rows, registry, metadata


def build_sequences(validated_rows):
    """{concept_id: {learner_id: [correct, ...]}} in attempt and pinned-question order.
    Expects the output of validate_answer_rows, which is already sorted that way."""
    sequences = {}
    for row in validated_rows:
        sequences.setdefault(row["concept_id"], {}).setdefault(row["learner_id"], []).append(row["correct"])
    return sequences


def _emission(params, correct):
    """P(observation | unknown), P(observation | known)."""
    if correct:
        return params["p_guess"], 1 - params["p_slip"]
    return 1 - params["p_guess"], params["p_slip"]


def _forward_backward(params, sequence):
    learn = params["p_learn"]
    emissions = [_emission(params, o) for o in sequence]
    alphas, scales = [], []
    alpha = ((1 - params["p_init"]) * emissions[0][0], params["p_init"] * emissions[0][1])
    for t in range(len(sequence)):
        if t:
            prev = alphas[-1]
            alpha = (prev[0] * (1 - learn) * emissions[t][0],
                     (prev[0] * learn + prev[1]) * emissions[t][1])
        scale = alpha[0] + alpha[1]
        alphas.append((alpha[0] / scale, alpha[1] / scale))
        scales.append(scale)
    betas = [(1.0, 1.0)] * len(sequence)
    for t in range(len(sequence) - 2, -1, -1):
        e, b = emissions[t + 1], betas[t + 1]
        betas[t] = (((1 - learn) * e[0] * b[0] + learn * e[1] * b[1]) / scales[t + 1],
                    e[1] * b[1] / scales[t + 1])
    return alphas, betas, emissions, sum(math.log(s) for s in scales)


def _clip(params):
    return {k: min(max(v, BKT_BOUNDS[k][0]), BKT_BOUNDS[k][1]) for k, v in params.items()}


def _em(sequences, params, iterations, tolerance):
    previous = -math.inf
    for _ in range(iterations):
        init = known_total = unknown_total = slips = guesses = learned = unknown_before = 0.0
        log_likelihood = 0.0
        for sequence in sequences:
            alphas, betas, emissions, ll = _forward_backward(params, sequence)
            log_likelihood += ll
            gammas = []
            for a, b in zip(alphas, betas):
                g0, g1 = a[0] * b[0], a[1] * b[1]
                gammas.append((g0 / (g0 + g1), g1 / (g0 + g1)))
            init += gammas[0][1]
            for t, (g, observed) in enumerate(zip(gammas, sequence)):
                unknown_total += g[0]
                known_total += g[1]
                slips += g[1] * (not observed)
                guesses += g[0] * observed
                if t + 1 < len(sequence):
                    e, b = emissions[t + 1], betas[t + 1]
                    stay = alphas[t][0] * (1 - params["p_learn"]) * e[0] * b[0]
                    learn = alphas[t][0] * params["p_learn"] * e[1] * b[1]
                    kept = alphas[t][1] * e[1] * b[1]
                    learned += learn / (stay + learn + kept)
                    unknown_before += g[0]
        params = _clip({
            "p_init": init / len(sequences),
            "p_learn": learned / unknown_before if unknown_before else params["p_learn"],
            "p_slip": slips / known_total if known_total else params["p_slip"],
            "p_guess": guesses / unknown_total if unknown_total else params["p_guess"],
        })
        if log_likelihood - previous < tolerance:
            break
        previous = log_likelihood
    return params, _forward_backward_total(params, sequences)


def _forward_backward_total(params, sequences):
    return sum(_forward_backward(params, s)[3] for s in sequences)


def fit_bkt(sequences, iterations=200, tolerance=1e-6):
    """Fit one concept's BKT parameters by EM (Baum-Welch) from several starting points.
    `sequences` is a list of per-learner lists of booleans. Returns (params, log-likelihood)."""
    if not isinstance(sequences, list) or not sequences:
        raise ValueError("expected a non-empty list of answer sequences")
    for sequence in sequences:
        if not isinstance(sequence, list) or not sequence or any(type(o) is not bool for o in sequence):
            raise ValueError("each sequence must be a non-empty list of booleans")
    starts = [
        {"p_init": 0.3, "p_learn": 0.1, "p_slip": 0.1, "p_guess": 0.2},
        {"p_init": 0.1, "p_learn": 0.3, "p_slip": 0.05, "p_guess": 0.3},
        {"p_init": 0.5, "p_learn": 0.05, "p_slip": 0.2, "p_guess": 0.1},
    ]
    return max((_em(sequences, start, iterations, tolerance) for start in starts), key=lambda fit: fit[1])


def predict_sequence(params, sequence):
    """One-step-ahead P(correct) before each answer, and P(known) after the last one."""
    _check_params(params)
    known, predictions = params["p_init"], []
    for observed in sequence:
        if type(observed) is not bool:
            raise ValueError("observations must be booleans")
        p_correct = known * (1 - params["p_slip"]) + (1 - known) * params["p_guess"]
        predictions.append(p_correct)
        posterior = (known * (1 - params["p_slip"]) / p_correct if observed
                     else known * params["p_slip"] / (1 - p_correct))
        known = posterior + (1 - posterior) * params["p_learn"]
    return predictions, known


def split_learners(learner_ids, test_share=0.2):
    """Deterministic learner-level split by SHA-256, so no learner is in both sets."""
    if not 0 < test_share < 1:
        raise ValueError("test_share must be between 0 and 1")
    train, test = [], []
    for learner in sorted(set(learner_ids)):
        bucket = int(hashlib.sha256(learner.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF
        (test if bucket < test_share else train).append(learner)
    if not train or not test:
        raise ValueError("split left one side empty; need more learners")
    return train, test


def _auc(labels, scores):
    positives = [s for l, s in zip(labels, scores) if l]
    negatives = [s for l, s in zip(labels, scores) if not l]
    if not positives or not negatives:
        return None
    wins = sum(1.0 if p > n else 0.5 if p == n else 0.0 for p in positives for n in negatives)
    return wins / (len(positives) * len(negatives))


def evaluate(params, sequences, baseline_rate):
    """Held-out next-answer metrics for BKT against a constant training-set rate."""
    labels, bkt, mastered = [], [], 0
    for sequence in sequences:
        predictions, final_known = predict_sequence(params, sequence)
        labels.extend(sequence)
        bkt.extend(predictions)
        mastered += final_known >= MASTERY_THRESHOLD

    def log_loss(scores):
        eps = 1e-9
        return -sum(math.log(max(eps, s if l else 1 - s)) for l, s in zip(labels, scores)) / len(labels)

    constant = [baseline_rate] * len(labels)
    return {
        "answers": len(labels),
        "bkt_log_loss": round(log_loss(bkt), 4),
        "baseline_log_loss": round(log_loss(constant), 4),
        "bkt_auc": None if _auc(labels, bkt) is None else round(_auc(labels, bkt), 4),
        "bkt_accuracy": round(sum((s >= 0.5) == l for l, s in zip(labels, bkt)) / len(labels), 4),
        "baseline_accuracy": round(sum((baseline_rate >= 0.5) == l for l in labels) / len(labels), 4),
        "share_mastered_at_end": round(mastered / len(sequences), 4),
    }


def export_draft_params(fits, evaluations, dataset_metadata):
    """Draft artifact for review. Synthetic: never load it as the app's mastery model."""
    return {
        "schemaVersion": 1,
        "model": "bkt",
        "label": LABEL,
        "synthetic": True,
        "productionApproved": False,
        "notForAppUse": "Fitted on synthetic learners; app priors and unlock rules are unchanged.",
        "source": {"seed": dataset_metadata["seed"], "learners": dataset_metadata["learners"],
                   "attempts": dataset_metadata["attempts"],
                   "contentVersion": dataset_metadata["content_version"]},
        "masteryThreshold": MASTERY_THRESHOLD,
        "concepts": {
            concept: {"pInit": round(p["p_init"], 4), "pLearn": round(p["p_learn"], 4),
                      "pSlip": round(p["p_slip"], 4), "pGuess": round(p["p_guess"], 4)}
            for concept, p in sorted(fits.items())
        },
        "heldOutEvaluation": evaluations,
    }

In [ ]:
import unittest


class BktFitTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        rows, cls.registry, cls.metadata = simulate_bkt_learners(n_learners=300, n_attempts=6, seed=11)
        cls.rows = validate_answer_rows(rows, cls.registry)
        cls.sequences = build_sequences(cls.rows)

    def test_simulated_rows_pass_the_input_contract(self):
        self.assertEqual(len(self.rows), 300 * 6 * 2)
        self.assertTrue(self.metadata["synthetic"])
        self.assertFalse(self.metadata["production_approved"])

    def test_sequences_follow_attempt_order(self):
        self.assertEqual(set(self.sequences), {"money-in", "money-out"})
        self.assertTrue(all(len(s) == 6 for c in self.sequences.values() for s in c.values()))

    def test_fit_recovers_known_parameters(self):
        for concept, truth in SYNTHETIC_TRUE_PARAMS.items():
            fitted, log_likelihood = fit_bkt(list(self.sequences[concept].values()))
            self.assertTrue(math.isfinite(log_likelihood))
            for name, value in truth.items():
                with self.subTest(concept=concept, param=name):
                    self.assertAlmostEqual(fitted[name], value, delta=0.08)

    def test_fit_stays_in_bounds_when_everyone_is_right(self):
        fitted, _ = fit_bkt([[True, True, True]] * 20)
        for name, (low, high) in BKT_BOUNDS.items():
            self.assertTrue(low <= fitted[name] <= high, name)

    def test_fit_handles_single_answers(self):
        fitted, _ = fit_bkt([[True], [False], [True]])
        self.assertEqual(set(fitted), set(BKT_BOUNDS))

    def test_fit_rejects_invalid_input(self):
        for invalid in [[], None, [[]], [[1, 0]], [["true"]], "TFT"]:
            with self.subTest(invalid=invalid), self.assertRaises(ValueError):
                fit_bkt(invalid)

    def test_prediction_rises_after_correct_answers(self):
        params = SYNTHETIC_TRUE_PARAMS["money-in"]
        predictions, known = predict_sequence(params, [True, True, True])
        self.assertLess(predictions[0], predictions[1])
        self.assertLess(predictions[1], predictions[2])
        self.assertGreater(known, params["p_init"])

    def test_prediction_rejects_invalid_params_and_observations(self):
        params = dict(SYNTHETIC_TRUE_PARAMS["money-in"])
        for bad in [{**params, "p_slip": 0.9}, {**params, "p_guess": True}, {"p_init": 0.5}]:
            with self.subTest(bad=bad), self.assertRaises(ValueError):
                predict_sequence(bad, [True])
        with self.assertRaises(ValueError):
            predict_sequence(params, [1])

    def test_split_keeps_learners_apart_and_is_deterministic(self):
        learners = [r["learner_id"] for r in self.rows]
        train, test = split_learners(learners)
        self.assertFalse(set(train) & set(test))
        self.assertEqual(len(train) + len(test), 300)
        self.assertEqual((train, test), split_learners(list(reversed(learners))))
        with self.assertRaises(ValueError):
            split_learners(learners, test_share=1)

    def test_held_out_bkt_beats_constant_baseline(self):
        train, test = split_learners([r["learner_id"] for r in self.rows])
        for concept, by_learner in self.sequences.items():
            train_seqs = [by_learner[l] for l in train]
            rate = sum(map(sum, train_seqs)) / sum(map(len, train_seqs))
            fitted, _ = fit_bkt(train_seqs)
            result = evaluate(fitted, [by_learner[l] for l in test], rate)
            with self.subTest(concept=concept):
                self.assertLess(result["bkt_log_loss"], result["baseline_log_loss"])
                self.assertGreater(result["bkt_auc"], 0.5)

    def test_export_is_labelled_synthetic_and_not_approved(self):
        fits = {c: fit_bkt(list(s.values()))[0] for c, s in self.sequences.items()}
        draft = export_draft_params(fits, {}, self.metadata)
        self.assertTrue(draft["synthetic"])
        self.assertFalse(draft["productionApproved"])
        self.assertIn("SYNTHETIC", draft["label"])
        self.assertEqual(set(draft["concepts"]), {"money-in", "money-out"})
        json.dumps(draft)

    def test_simulation_rejects_invalid_arguments(self):
        for kwargs in [{"seed": True}, {"n_learners": 0}, {"n_attempts": 1.5},
                       {"true_params": {"money-in": SYNTHETIC_TRUE_PARAMS["money-in"]}}]:
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                simulate_bkt_learners(**kwargs)


# Runs in the notebook; test_bkt_fit.py loads this class without running it here.
if __name__ == "__main__":
    fit_result = unittest.TextTestRunner(verbosity=1).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(BktFitTests))
    if not fit_result.wasSuccessful():
        raise AssertionError("SYNTHETIC BKT fitting tests failed")
    print("SYNTHETIC BKT fitting checks passed.")

In [ ]:
sim_rows, sim_registry, sim_metadata = simulate_bkt_learners(n_learners=400, n_attempts=6, seed=7)
sim_validated = validate_answer_rows(sim_rows, sim_registry)
sim_sequences = build_sequences(sim_validated)
train_ids, test_ids = split_learners([row["learner_id"] for row in sim_validated])
print(LABEL)
print(f"SYNTHETIC: {len(sim_validated)} answers; {len(train_ids)} train / {len(test_ids)} held-out learners")

fits, evaluations = {}, {}
for concept, by_learner in sorted(sim_sequences.items()):
    train_seqs = [by_learner[learner] for learner in train_ids]
    rate = sum(map(sum, train_seqs)) / sum(map(len, train_seqs))
    fits[concept], _ = fit_bkt(train_seqs)
    evaluations[concept] = evaluate(fits[concept], [by_learner[learner] for learner in test_ids], rate)
    print(f"\n{concept}")
    print(f"  {'param':<8} {'true':>6} {'fitted':>7}")
    for name in BKT_BOUNDS:
        print(f"  {name:<8} {SYNTHETIC_TRUE_PARAMS[concept][name]:>6.3f} {fits[concept][name]:>7.3f}")
    print("  held-out:", json.dumps(evaluations[concept]))

draft = export_draft_params(fits, evaluations, sim_metadata)
path = "knowledge-tracing-params.synthetic-draft.json"
with open(path, "w", encoding="utf-8") as handle:
    json.dump(draft, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
print(f"\nWrote {path} (SYNTHETIC draft; do not load it in the app).")
try:
    from google.colab import files
    files.download(path)
except ImportError:
    pass

## Next increment — not implemented

Later: fit on real quiz answers exported by R3 (outside this shared notebook), review the parameters, then port `predict_sequence` to `packages/engine` and run it on device. Initial app mastery priors are none=0.1, some=0.3, regular=0.5; weak <0.6, practising >=0.6 and <0.95, mastered >=0.95. These rules are recorded only, not applied in this notebook. Fitted synthetic parameters must not automatically replace app priors or influence production recommendations.

Do not describe any synthetic result as real-student accuracy.